In [3]:
import os
import string
import math

class Libro:
    def __init__(self, titulo: str, palabras: list[str]):
        self.titulo = titulo
        self.palabras = palabras  # Lista de palabras del libro ya limpias


class Recomendador:
    def __init__(self, libros: list[Libro]) -> None:
        """
        libros: lista con instancias de tipo `Libro`
        """
        self.libros = libros
        self._pesos = None  # Se calcularan con un setter (ver `set_pesos`)
        
        # Vocabulario único de todos los libros combinados
        self.vocabulario = list(set(palabra for libro in libros for palabra in libro.palabras))

    def set_pesos(self) -> None:
        """Calcula los pesos del algorítmo TF-IDF requeridos para las
        recomendaciones y los guarda en `self._pesos`
        """
        num_libros = len(self.libros)
        
        # 1. Calcular el DF (Document Frequency): En cuántos libros aparece cada palabra
        df = {palabra: 0 for palabra in self.vocabulario}
        for libro in self.libros:
            palabras_unicas = set(libro.palabras)
            for palabra in palabras_unicas:
                if palabra in df:
                    df[palabra] += 1

        # 2. Calcular el IDF para cada palabra (usando logaritmo natural)
        idf = {}
        for palabra, conteo in df.items():
            idf[palabra] = math.log(num_libros / conteo)

        # 3. Calcular la matriz de pesos TF-IDF para cada libro
        self._pesos = []
        for libro in self.libros:
            tf = {}
            for palabra in libro.palabras:
                tf[palabra] = tf.get(palabra, 0) + 1
            
            pesos_libro = {}
            for palabra in self.vocabulario:
                freq = tf.get(palabra, 0)
                if freq > 0:
                    pesos_libro[palabra] = freq * idf[palabra]
                else:
                    pesos_libro[palabra] = 0.0
                    
            self._pesos.append(pesos_libro)

    def get_pesos(self):
        """Regresa los pesos calculados"""
        return self._pesos

    def _producto_punto(self, idx_1: int, idx_2: int) -> float:
        """Producto punto entre los libros con índices idx_1 y idx_2."""
        pesos_1 = self._pesos[idx_1]
        pesos_2 = self._pesos[idx_2]
        
        prod_punto = 0.0
        for palabra in self.vocabulario:
            prod_punto += pesos_1[palabra] * pesos_2[palabra]
        return prod_punto

    def _similitud(self, idx_1, idx_2) -> float:
        """Similitud entre los libros con índices idx_1 y idx_2 de acuerdo al
        coseno del ángulo que forman sus vectores.
        """
        if idx_1 == idx_2:
            return 1.0
            
        prod_punto = self._producto_punto(idx_1, idx_2)
        
        norma_1 = math.sqrt(sum(peso ** 2 for peso in self._pesos[idx_1].values()))
        norma_2 = math.sqrt(sum(peso ** 2 for peso in self._pesos[idx_2].values()))
        
        if norma_1 == 0 or norma_2 == 0:
            return 0.0
            
        return prod_punto / (norma_1 * norma_2)

    def mostrar_libros(self):
        """Mostrarle al usuario el índice y nombre para cada libro de acuerdo a
        nuestra lista de libros `self.libros`.
        """
        print(f"{'Índice':<8} | {'Título del Libro'}")
        print("-" * 50)
        for idx, libro in enumerate(self.libros):
            print(f"{idx:<8} | {libro.titulo}")

    def resumen(self, idx_libro, num_palabras) -> list[str]:
        """Regresa una lista con las palabras más representativas de un libro"""
        pesos_libro = self._pesos[idx_libro]
        palabras_ordenadas = sorted(pesos_libro.items(), key=lambda x: x[1], reverse=True)
        top_palabras = [palabra for palabra, peso in palabras_ordenadas if peso > 0]
        return top_palabras[:num_palabras]

    def libros_similares(self, idx_libro, num_libros) -> list[str]:
        """Regresa una lista con los libros más parecidos a un libro dado."""
        similitudes = []
        for idx_otro in range(len(self.libros)):
            if idx_otro != idx_libro:
                sim = self._similitud(idx_libro, idx_otro)
                similitudes.append((idx_otro, sim))
                
        similitudes_ordenadas = sorted(similitudes, key=lambda x: x[1], reverse=True)
        
        recomendaciones = []
        for i in range(min(num_libros, len(similitudes_ordenadas))):
            idx_recom, score = similitudes_ordenadas[i]
            titulo_libro = self.libros[idx_recom].titulo
            recomendaciones.append(f"{titulo_libro} (Similitud: {score:.4f})")
            
        return recommendaciones

In [4]:
ruta_carpeta = "./Books"
mis_libros = []

# Listamos y ordenamos los archivos de tus 50 libros descargados
archivos = sorted([f for f in os.listdir(ruta_carpeta) if f.endswith(".txt")])

for archivo in archivos:
    ruta_archivo = os.path.join(ruta_carpeta, archivo)
    
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        texto = f.read().lower()
        
        # Quitamos signos de puntuación básicos
        texto_limpio = texto.translate(str.maketrans("", "", string.punctuation))
        palabras = texto_limpio.split()
        
        titulo_libro = archivo.replace(".txt", "").replace("_", " ")
        
        nuevo_libro = Libro(titulo=titulo_libro, palabras=palabras)
        mis_libros.append(nuevo_libro)

print(f"¡Listo! Se procesaron correctamente {len(mis_libros)} libros para el recomendador.")

¡Listo! Se procesaron correctamente 85 libros para el recomendador.


In [5]:
# Inicializamos el recomendador con tus libros cargados
rec = Recomendador(mis_libros)

# Calculamos los pesos TF-IDF (va a tardar unos segundos)
rec.set_pesos()

# Mostramos la lista completa para verificar que todo esté en orden
rec.mostrar_libros()

Índice   | Título del Libro
--------------------------------------------------
0        | A Room with a View by E. M.  Forster (2469)
1        | A Room with a View by E. M.  Forster (2907)
2        | Alice's Adventures in Wonderland by Lewis Carroll (2470)
3        | Alice's Adventures in Wonderland by Lewis Carroll (2648)
4        | An anthology of German literature by Calvin Thomas (1734)
5        | Bidwell's Travels, from Wall Street to London Prison: Fifteen Years in Solitude by Austin Bidwell (1349)
6        | Buddenbrooks: Verfall einer Familie by Thomas Mann (1301)
7        | Calculus Made Easy by Silvanus P.  Thompson (6549)
8        | Chambers's Twentieth Century Dictionary (part 1 of 4: A-D) (1328)
9        | Chambers's Twentieth Century Dictionary (part 1 of 4: A-D) (2542)
10       | Chambers's Twentieth Century Dictionary (part 1 of 4  A-D) (2542)
11       | Concrete Construction: Methods and Costs by Halbert Powers Gillette and Charles Shattuck Hill (2102)
12       | Cranf

In [4]:
# Probamos ver el resumen de palabras clave del libro con índice 0
print("Palabras clave del libro 0:")
print(rec.resumen(idx_libro=0, num_palabras=10))

print("\n" + "="*40 + "\n")

# Probamos pedir 3 recomendaciones parecidas al libro 0
print("Recomendaciones para el libro 0:")
print(rec.libros_similares(idx_libro=0, num_libros=3))

Palabras clave del libro 0:
['lucy', 'beebe', 'honeychurch', 'cecil', 'bartlett', 'freddy', 'emerson', '“i', 'vyse', 'emersons']


Recomendaciones para el libro 0:


NameError: name 'recommendations' is not defined